In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [4]:
df = pd.read_csv("train.csv")
df.head()

,Unnamed: 0,Name,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,New_Price,Price
0,1,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,Diesel,Manual,First,19.67 kmpl,1582 CC,126.2 bhp,5.0,NaN,12.50
1,2,Honda Jazz V,Chennai,2011,46000,Petrol,Manual,First,13 km/kg,1199 CC,88.7 bhp,5.0,8.61 Lakh,4.50
2,3,Maruti Ertiga VDI,Chennai,2012,87000,Diesel,Manual,First,20.77 kmpl,1248 CC,88.76 bhp,7.0,NaN,6.00
3,4,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,Diesel,Automatic,Second,15.2 kmpl,1968 CC,140.8 bhp,5.0,NaN,17.74
4,6,Nissan Micra Diesel XV,Jaipur,2013,86999,Diesel,Manual,First,23.08 kmpl,1461 CC,63.1 bhp,5.0,NaN,3.50


In [7]:
print(df.shape)
print(df.columns.tolist())
df.info()

(5847, 14)
['Unnamed: 0', 'Name', 'Location', 'Year', 'Kilometers_Driven', 'Fuel_Type', 'Transmission', 'Owner_Type', 'Mileage', 'Engine', 'Power', 'Seats', 'New_Price', 'Price']
<class 'pandas.DataFrame'>
RangeIndex: 5847 entries, 0 to 5846
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         5847 non-null   int64  
 1   Name               5847 non-null   str    
 2   Location           5847 non-null   str    
 3   Year               5847 non-null   int64  
 4   Kilometers_Driven  5847 non-null   int64  
 5   Fuel_Type          5847 non-null   str    
 6   Transmission       5847 non-null   str    
 7   Owner_Type         5847 non-null   str    
 8   Mileage            5845 non-null   str    
 9   Engine             5811 non-null   str    
 10  Power              5811 non-null   str    
 11  Seats              5809 non-null   float64
 12  New_Price          815 non-null    str    
 13  

In [9]:
missing_values = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100

missing_summary = pd.DataFrame({
    "Missing Count": missing_values,
    "Missing Percentage": missing_percent
})
print(missing_summary)

                   Missing Count  Missing Percentage
Unnamed: 0                     0            0.000000
Name                           0            0.000000
Location                       0            0.000000
Year                           0            0.000000
Kilometers_Driven              0            0.000000
Fuel_Type                      0            0.000000
Transmission                   0            0.000000
Owner_Type                     0            0.000000
Mileage                        2            0.034206
Engine                        36            0.615700
Power                         36            0.615700
Seats                         38            0.649906
New_Price                   5032           86.061228
Price                          0            0.000000


In [10]:
for col in ["Mileage", "Engine", "Power"]:
    df[col] = pd.to_numeric(
        df[col].str.extract(r"(\d+\.?\d*)")[0],
        errors="coerce"
    )

print(df[["Mileage", "Engine", "Power"]].isnull().sum())

Mileage     2
Engine     36
Power      36
dtype: int64


In [11]:
for col in ["Mileage", "Engine", "Power"]:
    df[col] = df[col].fillna(df[col].median())

df["Seats"] = df["Seats"].fillna(df["Seats"].mode()[0])

print(df.isnull().sum())

Unnamed: 0              0
Name                    0
Location                0
Year                    0
Kilometers_Driven       0
Fuel_Type               0
Transmission            0
Owner_Type              0
Mileage                 0
Engine                  0
Power                   0
Seats                   0
New_Price            5032
Price                   0
dtype: int64


In [12]:
df["New_Price"] = pd.to_numeric(
    df["New_Price"].str.extract(r"(\d+\.?\d*)")[0],
    errors="coerce"
)

df = df.drop(columns=["New_Price"])

print(df.isnull().sum())

Unnamed: 0           0
Name                 0
Location             0
Year                 0
Kilometers_Driven    0
Fuel_Type            0
Transmission         0
Owner_Type           0
Mileage              0
Engine               0
Power                0
Seats                0
Price                0
dtype: int64


### Q1-(a): Missing Value Handling

The dataset contains 5,847 observations and 14 columns. Missing values were identified in five columns: Mileage (2), Engine (36), Power (36), Seats (38), and New_Price (5,032).

For Mileage, Engine, and Power, the units were removed and the values were converted to numeric format. Missing values were then replaced with the median because the median is less sensitive to outliers.

Missing values in Seats were replaced with the mode because the number of seats is a discrete variable.

New_Price was removed after extracting its numerical values because approximately 86.06% of its observations were missing. Imputing such a large proportion of missing values could introduce substantial bias.

In [13]:
print(df[["Mileage", "Engine", "Power"]].head(10))

print(df[["Mileage", "Engine", "Power"]].dtypes)

   Mileage  Engine   Power
0    19.67  1582.0  126.20
1    13.00  1199.0   88.70
2    20.77  1248.0   88.76
3    15.20  1968.0  140.80
4    23.08  1461.0   63.10
5    11.36  2755.0  171.50
6    20.54  1598.0  103.60
7    22.30  1248.0   74.00
8    21.56  1462.0  103.25
9    16.80  1497.0  116.30
Mileage    float64
Engine     float64
Power      float64
dtype: object


In [15]:
print(df["Fuel_Type"].unique())

print(df["Transmission"].unique())

<StringArray>
['Diesel', 'Petrol', 'Electric']
Length: 3, dtype: str
<StringArray>
['Manual', 'Automatic']
Length: 2, dtype: str


In [16]:
df = pd.get_dummies(
    df,
    columns=["Fuel_Type", "Transmission"],
    dtype=int
)

encoded_columns = [
    col for col in df.columns
    if col.startswith(("Fuel_Type_", "Transmission_"))
]

print(df[encoded_columns].head())

   Fuel_Type_Diesel  Fuel_Type_Electric  Fuel_Type_Petrol  \
0                 1                   0                 0   
1                 0                   0                 1   
2                 1                   0                 0   
3                 1                   0                 0   
4                 1                   0                 0   

   Transmission_Automatic  Transmission_Manual  
0                       0                    1  
1                       0                    1  
2                       0                    1  
3                       1                    0  
4                       0                    1  


In [17]:
print("Dataset Shape:", df.shape)

print(df.columns.tolist())

print(df.filter(like="Fuel_Type_").sum(axis=1).value_counts())

print(df.filter(like="Transmission_").sum(axis=1).value_counts())

Dataset Shape: (5847, 16)
['Unnamed: 0', 'Name', 'Location', 'Year', 'Kilometers_Driven', 'Owner_Type', 'Mileage', 'Engine', 'Power', 'Seats', 'Price', 'Fuel_Type_Diesel', 'Fuel_Type_Electric', 'Fuel_Type_Petrol', 'Transmission_Automatic', 'Transmission_Manual']
1    5847
Name: count, dtype: int64
1    5847
Name: count, dtype: int64


In [18]:
df["Car_Age"] = 2026 - df["Year"]

print(df[["Year", "Car_Age"]].head(10))

   Year  Car_Age
0  2015       11
1  2011       15
2  2012       14
3  2013       13
4  2013       13
5  2016       10
6  2013       13
7  2012       14
8  2018        8
9  2012       14


In [23]:
#select
selected_df = df[["Name", "Year", "Car_Age", "Power", "Engine"]]
selected_df.head()

,Name,Year,Car_Age,Power,Engine
0,Hyundai Creta 1.6 CRDi SX Option,2015,11,126.20,1582.0
1,Honda Jazz V,2011,15,88.70,1199.0
2,Maruti Ertiga VDI,2012,14,88.76,1248.0
3,Audi A4 New 2.0 TDI Multitronic,2013,13,140.80,1968.0
4,Nissan Micra Diesel XV,2013,13,63.10,1461.0


In [25]:
#filter
filtered_df = df.loc[df["Price"] > 10]
filtered_df.head()

,Unnamed: 0,Name,Location,Year,Kilometers_Driven,Owner_Type,Mileage,Engine,Power,Seats,Price,Fuel_Type_Diesel,Fuel_Type_Electric,Fuel_Type_Petrol,Transmission_Automatic,Transmission_Manual,Car_Age
0,1,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,First,19.67,1582.0,126.2,5.0,12.50,1,0,0,0,1,11
3,4,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,Second,15.20,1968.0,140.8,5.0,17.74,1,0,0,1,0,13
5,7,Toyota Innova Crysta 2.8 GX AT 8S,Mumbai,2016,36000,First,11.36,2755.0,171.5,8.0,17.50,1,0,0,1,0,10
11,13,Land Rover Range Rover 2.2L Pure,Delhi,2014,72000,First,12.70,2179.0,187.7,5.0,27.00,1,0,0,1,0,12
12,14,Land Rover Freelander 2 TD4 SE,Pune,2012,85000,Second,0.00,2179.0,115.0,5.0,17.50,1,0,0,1,0,14


In [26]:
# rename
renamed_df = df.rename(columns={"Price": "Used_Car_Price"})

renamed_df[["Name", "Used_Car_Price"]].head()

,Name,Used_Car_Price
0,Hyundai Creta 1.6 CRDi SX Option,12.50
1,Honda Jazz V,4.50
2,Maruti Ertiga VDI,6.00
3,Audi A4 New 2.0 TDI Multitronic,17.74
4,Nissan Micra Diesel XV,3.50


In [27]:
# mutate
mutated_df = df.assign(
    Kilometers_Per_Year = df["Kilometers_Driven"] / df["Car_Age"].replace(0, np.nan)
)

mutated_df[["Kilometers_Driven", "Car_Age", "Kilometers_Per_Year"]].head()

,Kilometers_Driven,Car_Age,Kilometers_Per_Year
0,41000,11,3727.272727
1,46000,15,3066.666667
2,87000,14,6214.285714
3,40670,13,3128.461538
4,86999,13,6692.230769


In [28]:
# arrange
arranged_df = df.sort_values(
    by="Price",
    ascending=False
)

arranged_df[["Name", "Price"]].head(10)

,Name,Price
3952,Land Rover Range Rover 3.0 Diesel LWB Vogue,160.00
5620,Lamborghini Gallardo Coupe,120.00
5752,Jaguar F Type 5.0 V8 S,100.00
1457,Land Rover Range Rover Sport SE,97.07
1917,BMW 7 Series 740Li,93.67
1927,BMW 7 Series 740Li,93.00
4556,Mercedes-Benz SLK-Class 55 AMG,90.00
5380,BMW X6 xDrive 40d M Sport,85.00
2035,Mercedes-Benz SLC 43 AMG,83.96
1830,Mercedes-Benz GLS 350d Grand Edition,79.00


In [29]:
# group by
location_summary = (
    df.groupby("Location", as_index=False)
    .agg(
        Average_Price=("Price", "mean"),
        Max_Price=("Price", "max"),
        Car_Count=("Price", "count")
    )
)

location_summary.head(10)

,Location,Average_Price,Max_Price,Car_Count
0,Ahmedabad,8.567248,85.00,218
1,Bangalore,13.482670,93.00,352
2,Chennai,7.958340,70.00,476
3,Coimbatore,15.160206,93.67,631
4,Delhi,9.881944,120.00,540
5,Hyderabad,9.997423,160.00,710
6,Jaipur,5.916725,42.50,403
7,Kochi,11.309109,97.07,640
8,Kolkata,5.733924,75.00,525
9,Mumbai,9.592546,75.00,762
